# Multi-Task BERT Model for Yelp Reviews

This notebook implements a **multi-task model** using BERT embeddings to predict:

1. **Sentiment**: Negative (1–2 stars), Neutral (3 stars), Positive (4–5 stars)  
2. **Star Rating**: 1–5 stars

The model uses a **shared BERT encoder** followed by a fully connected hidden layer, then **separate heads** for sentiment and star rating.  

### Features:
- Pretrained `bert-base-uncased` embeddings
- Multi-task learning
- Cross-entropy loss for both tasks
- Evaluation metrics: Accuracy and Macro-F1

In [ ]:
import os
import glob
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# Load data
def load_data():
    folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
    csv_files = glob.glob(os.path.join(folder, "*.csv"))
    
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df['state'] = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df)
    
    data = pd.concat(dfs, ignore_index=True)
    df = data.dropna()
    
    # Sentiment: 0=negative (1-2), 1=neutral (3), 2=positive (4-5)
    df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
    
    # Star rating labels 0-4 (for CrossEntropy)
    df['star_label'] = df['stars'] - 1
    
    print(f"Loaded {len(df)} reviews from {len(dfs)} states.")
    return df

In [ ]:
# Tokenize dataset
MAX_LEN = 128
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class YelpDataset(Dataset):
    def __init__(self, texts, sentiment_labels, star_labels):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.sentiment_labels = torch.tensor(sentiment_labels.tolist(), dtype=torch.long)
        self.star_labels = torch.tensor(star_labels.tolist(), dtype=torch.long)
    
    def __len__(self):
        return len(self.sentiment_labels)
    
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'sentiment_labels': self.sentiment_labels[idx],
            'star_labels': self.star_labels[idx]
        }

In [ ]:
# Multi-Task model
class MultiTaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.fc_shared = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sentiment_head = nn.Linear(256, 3)
        self.star_head = nn.Linear(256, 5)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(self.relu(self.fc_shared(cls_output)))
        sentiment_logits = self.sentiment_head(x)
        star_logits = self.star_head(x)
        return sentiment_logits, star_logits

In [ ]:
# Training
def train(model, dataloader, optimizer, criterion_sent, criterion_star, device):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        sentiment_labels = batch['sentiment_labels'].to(device)
        star_labels = batch['star_labels'].to(device)
        
        optimizer.zero_grad()
        sentiment_logits, star_logits = model(input_ids, attention_mask)
        loss_sent = criterion_sent(sentiment_logits, sentiment_labels)
        loss_star = criterion_star(star_logits, star_labels)
        loss = loss_sent + loss_star
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


In [ ]:
# Evaluation
def evaluate(model, dataloader, device):
    model.eval()
    all_sent_preds = []
    all_sent_labels = []
    all_star_preds = []
    all_star_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sentiment_labels = batch['sentiment_labels'].to(device)
            star_labels = batch['star_labels'].to(device)
            
            sentiment_logits, star_logits = model(input_ids, attention_mask)
            all_sent_preds.extend(torch.argmax(sentiment_logits, dim=1).cpu().numpy())
            all_sent_labels.extend(sentiment_labels.cpu().numpy())
            all_star_preds.extend(torch.argmax(star_logits, dim=1).cpu().numpy())
            all_star_labels.extend(star_labels.cpu().numpy())
    
    # Metrics
    sent_acc = accuracy_score(all_sent_labels, all_sent_preds)
    star_acc = accuracy_score(all_star_labels, all_star_preds)
    return sent_acc, star_acc

In [ ]:
if __name__ == "__main__":
    df = load_data()
    
    X_train, X_test, y_sent_train, y_sent_test, y_star_train, y_star_test = train_test_split(
        df['clean_text'], df['sentiment'], df['star_label'], test_size=0.1, random_state=42, stratify=df['sentiment']
    )
    
    train_dataset = YelpDataset(X_train, y_sent_train, y_star_train)
    test_dataset = YelpDataset(X_test, y_sent_test, y_star_test)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MultiTaskModel().to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    criterion_sent = nn.CrossEntropyLoss()
    criterion_star = nn.CrossEntropyLoss()
    
    for epoch in range(3):
        loss = train(model, train_loader, optimizer, criterion_sent, criterion_star, device)
        sent_acc, star_acc = evaluate(model, test_loader, device)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Sentiment Acc={sent_acc:.4f}, Star Acc={star_acc:.4f}")
